In [ ]:
!nvidia-smi

Thu Feb 26 13:17:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U bitsandbytes
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 30.0 MB/s eta 0:00:00


In [ ]:
import os
import sys
import torch
from transformers import AutoTokenizer, GenerationConfig, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

model_name = 'RefalMachine/RuadaptQwen3-4B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    #quantization_config=bnb_config, # Раскомментировать в случае использования моделей побольше, где не нужна квантизация
    device_map="cuda:0", # В случае нескольких GPU нужно указать конкретную, cuda:0, например
    attn_implementation="sdpa", #attn_implementation='flash_attention_2' для GPU, которые поддерживают
)
model.eval()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(146260, 2560, padding_idx=146213)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (

In [ ]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(model_name)
generation_config

GenerationConfig {
  "do_sample": true,
  "eos_token_id": 146215,
  "pad_token_id": 146213,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

In [ ]:
generation_config.temperature = 0.3
generation_config.max_new_tokens = 512
generation_config

GenerationConfig {
  "do_sample": true,
  "eos_token_id": 146215,
  "max_new_tokens": 512,
  "pad_token_id": 146213,
  "temperature": 0.3,
  "top_k": 20,
  "top_p": 0.8
}

In [ ]:
print(tokenizer.apply_chat_template([{'role': 'user', 'content': 'Что такое ИИ?'}], tokenize=False, add_generation_prompt=True))

<|im_start|>user
Что такое ИИ?<|im_end|>
<|im_start|>assistant



In [ ]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- for message in messages %}
    {%- if (message.ro

In [ ]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Что такое ИИ?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
output

'**ИИ — это Искусственный интеллект** (от англ. *Artificial Intelligence*, сокращённо AI).\n\nЭто область науки о создании компьютерных систем и программ, способных выполнять задачи, которые традиционно требуют человеческого интеллекта. Такие задачи включают:\n\n- Обработку естественного языка (понимание и генерация текста, чат-боты),\n- Распознавание образов (распознавание лиц, анализ изображений),\n- Решение сложных логических задач,\n- Обучение на основе опыта (машинное обучение),\n- Планирование и принятие решений,\n- Работа с числами и данными.\n\n### Как работает ИИ?\nИИ может быть реализован разными способами:\n- **Нейросети** — имитируют работу человеческого мозга, особенно в задачах распознавания и прогнозирования.\n- **Машинное обучение (ML)** — система учится на данных, не зная заранее, как решать задачу.\n- **Обучение с подкреплением** — ИИ учится "играть", "двигаться" или принимать решения, получая награды за правильные действия.\n- **Обработка естественного языка (NLP)** 

In [ ]:
print(output)

**ИИ — это Искусственный интеллект** (от англ. *Artificial Intelligence*, сокращённо AI).

Это область науки о создании компьютерных систем и программ, способных выполнять задачи, которые традиционно требуют человеческого интеллекта. Такие задачи включают:

- Обработку естественного языка (понимание и генерация текста, чат-боты),
- Распознавание образов (распознавание лиц, анализ изображений),
- Решение сложных логических задач,
- Обучение на основе опыта (машинное обучение),
- Планирование и принятие решений,
- Работа с числами и данными.

### Как работает ИИ?
ИИ может быть реализован разными способами:
- **Нейросети** — имитируют работу человеческого мозга, особенно в задачах распознавания и прогнозирования.
- **Машинное обучение (ML)** — система учится на данных, не зная заранее, как решать задачу.
- **Обучение с подкреплением** — ИИ учится "играть", "двигаться" или принимать решения, получая награды за правильные действия.
- **Обработка естественного языка (NLP)** — позволяет ИИ по